# 01 · Explore the data and set up a baseline

**Question:** Do defensive backs who change direction more efficiently in Combine drills stay tighter on receivers in NFL games, and does that depend on throw depth?

This notebook takes a first pass at the data: inventory the files, confirm defensive back coverage, and set up the feature and target pipelines. The early cells inspect the files directly rather than hard-coding a schema, and the configuration blocks in sections 4 and 5 map the pipelines onto the dataset's column names.

**Sections**
1. Find and load the data
2. Inventory every file
3. Confirm defensive backs are present
4. Combine movement features
5. In-game coverage tightness
6. Baseline comparison

## 1. Find and load the data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# Works both locally (repo's data/ folder) and on Kaggle (/kaggle/input/...)
CANDIDATE_DIRS = [
    Path("/kaggle/input/nfl-big-data-bowl-2027"),
    Path("../data"),
    Path("data"),
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists() and any(p.rglob("*.csv"))), None)

if DATA_DIR is None:
    print("No competition files found in data/ or /kaggle/input.")
    files = []
else:
    files = sorted(DATA_DIR.rglob("*.csv"))
    print(f"Data folder: {DATA_DIR.resolve()}")
    for f in files:
        print(f"  {f.relative_to(DATA_DIR)}  ({f.stat().st_size / 1e6:,.1f} MB)")

## 2. Inventory every file

For each file: columns and a few sample rows. Tracking files are large, so each is previewed from its first 5,000 rows.

In [ ]:
PREVIEW_ROWS = 5_000
previews = {}

for f in files:
    df = pd.read_csv(f, nrows=PREVIEW_ROWS, low_memory=False)
    previews[f.stem] = df
    print("=" * 80)
    print(f"{f.name}: {len(df.columns)} columns (showing first {min(len(df), PREVIEW_ROWS):,} rows)")
    print(list(df.columns))
    display(df.head(3))

### Locating key fields

A quick keyword search across all files for the fields this analysis depends on: player IDs, positions, drills, movement, and coordinates.

In [ ]:
def find_columns(*keywords):
    """Return {file: [matching columns]} for columns containing any keyword (case-insensitive)."""
    hits = {}
    for name, df in previews.items():
        cols = [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]
        if cols:
            hits[name] = cols
    return hits

for label, kws in {
    "position": ["position", "pos"],
    "player id": ["nflid", "playerid", "player_id", "gsis"],
    "drill / event": ["drill", "event", "test"],
    "speed & accel": ["speed", "accel", "decel", "dist", "dir", "orient"],
    "coordinates": ["x", "y"],
}.items():
    print(f"{label}:", find_columns(*kws) or "none found")

## 3. Confirm defensive backs are present

In [ ]:
# Common labels for defensive backs; the dataset may also use a grouped label like "DB".
DB_LABELS = {"CB", "S", "FS", "SS", "DB", "NB"}

for name, df in previews.items():
    for col in [c for c in df.columns if "position" in c.lower() or c.lower() in {"pos", "positiongroup"}]:
        counts = df[col].value_counts()
        print(f"{name}.{col}:")
        print(counts.to_string())
        db_rows = counts[counts.index.astype(str).str.upper().isin(DB_LABELS)].sum()
        print(f"  -> defensive back rows in preview: {db_rows}\n")

## 4. Combine movement features

For each defensive back, I want to summarize *how* he moves in change-of-direction drills, not just how fast he runs in a straight line.

Features:
- **Peak deceleration:** how hard he can brake.
- **Peak re-acceleration:** how quickly he gets back up to speed after a direction change.
- **Speed retained through the turn:** speed at the cut relative to speed entering it.
- **Top speed:** straight-line speed, standing in for the 40-yard dash as the baseline.

In [ ]:
# Dataset column mapping (from the inventory in sections 2-3)
COMBINE_FILE = None        # e.g. "combine_tracking"
PLAYER_COL   = None        # player identifier
DRILL_COL    = None        # which drill each row belongs to
TIME_COL     = None        # timestamp / frame
SPEED_COL    = None        # speed
ACCEL_COL    = None        # acceleration (signed, if available)
CHANGE_OF_DIRECTION_DRILLS = []   # drill names that involve turning

def combine_features(df):
    """One row per player of movement traits. Assumes rows are sorted by time within player+drill."""
    df = df.sort_values([PLAYER_COL, DRILL_COL, TIME_COL])
    cod = df[df[DRILL_COL].isin(CHANGE_OF_DIRECTION_DRILLS)]

    feats = pd.DataFrame({
        "top_speed": df.groupby(PLAYER_COL)[SPEED_COL].max(),
        "peak_decel": cod.groupby(PLAYER_COL)[ACCEL_COL].min().abs(),
        "peak_accel": cod.groupby(PLAYER_COL)[ACCEL_COL].max(),
    })

    # Speed retained through the turn: the slowest point in a drill relative to the speed just before it
    def retained(g):
        s = g[SPEED_COL].to_numpy()
        if len(s) < 5:
            return np.nan
        i = int(np.argmin(s[2:-2])) + 2
        before = s[max(0, i - 5):i].max()
        return s[i] / before if before > 0 else np.nan

    feats["speed_retained"] = cod.groupby([PLAYER_COL, DRILL_COL]).apply(retained).groupby(level=0).mean()
    return feats.reset_index()

if COMBINE_FILE:
    combine_df = pd.read_csv(DATA_DIR / f"{COMBINE_FILE}.csv", low_memory=False)
    db_features = combine_features(combine_df)
    display(db_features.describe())
else:
    print("Combine column mapping not set.")

## 5. In-game coverage tightness

The target: **how close is the defender to his receiver when the ball arrives?** Smaller distance means tighter coverage.

Method:
1. Identify passing plays and the frame where the ball arrives.
2. On that frame, compute each defensive back's distance to the nearest offensive player.
3. Bucket the throw by depth: short (under ~9 m), intermediate, or deep (over ~18 m) downfield.
4. Average by player and depth band, keeping play counts so small samples can be weighted appropriately.

In [ ]:
# Dataset column mapping (from the inventory in sections 2-3)
GAME_FILE      = None   # in-game tracking file
GAME_ID_COL    = None
PLAY_ID_COL    = None
FRAME_COL      = None
EVENT_COL      = None   # column with event labels
ARRIVAL_EVENTS = []     # e.g. ["pass_arrived"]
SIDE_COL       = None   # offense/defense indicator
X_COL, Y_COL   = None, None

def nearest_receiver_distance(frame_df, db_mask, off_mask):
    """For each defender on one frame, the distance to the closest offensive player."""
    d = frame_df[db_mask][[PLAYER_COL, X_COL, Y_COL]].to_numpy()
    o = frame_df[off_mask][[X_COL, Y_COL]].to_numpy()
    if len(d) == 0 or len(o) == 0:
        return pd.DataFrame(columns=[PLAYER_COL, "separation"])
    dist = np.sqrt(((d[:, None, 1:].astype(float) - o[None, :, :].astype(float)) ** 2).sum(-1)).min(1)
    return pd.DataFrame({PLAYER_COL: d[:, 0], "separation": dist})

if not GAME_FILE:
    print("Game-tracking column mapping not set.")

## 6. Baseline comparison

The core test: **do movement traits explain coverage tightness better than straight-line speed alone?**

- Baseline: `separation ~ top_speed`
- Movement model: `separation ~ top_speed + peak_decel + speed_retained`

Each Combine class is small, so I compare models with cross-validation rather than a single train/test split, and repeat the comparison within each depth band. Players are weighted by targeted snaps so a defender with 3 targets doesn't count the same as one with 300.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

def compare_models(df, target="separation", weight_col=None):
    base = ["top_speed"]
    full = ["top_speed", "peak_decel", "speed_retained"]
    df = df.dropna(subset=full + [target])
    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    out = {}
    for name, cols in {"baseline": base, "movement": full}.items():
        scores = cross_val_score(LinearRegression(), df[cols], df[target], cv=cv, scoring="r2")
        out[name] = (scores.mean(), scores.std())
    return pd.DataFrame(out, index=["mean_r2", "sd_r2"]).T

## Next steps

- Plot each Combine feature against coverage tightness, split by throw depth
- Highlight a few illustrative players, such as high-quickness defenders who outperformed their draft slot
- Animate representative plays contrasting tight and loose coverage
- Summarize the takeaway for scouts in plain language